In [ ]:
!pip install torch torchaudio soundfile datasets

In [ ]:
import torch
import torchaudio
import os
import random
import soundfile as sf
from datasets import load_dataset, Audio

print("Torch version:", torch.__version__)
print("Torchaudio version:", torchaudio.__version__)

In [ ]:
from datasets import load_dataset

# Load waveform/audio data
ds_audio = load_dataset("szzs1693/coswara-data", "audio", split="train")

print(ds_audio[0])

In [ ]:
from datasets import load_dataset

# Audio (waveforms)
ds_audio = load_dataset("szzs1693/coswara-data", "audio", split="train")

# Metadata (labels, participant info)
ds_meta = load_dataset("szzs1693/coswara-data", "metadata", split="train")

In [ ]:
# Shuffle and pick 200 random indices
ds_audio_shuffled = ds_audio.shuffle(seed=42)
subset_indices = list(range(200))  # first 200 samples

ds_audio_small = ds_audio_shuffled.select(subset_indices)
ds_meta_small = ds_meta.select(subset_indices)  # aligned

print("Subset audio size:", len(ds_audio_small))
print("Subset metadata size:", len(ds_meta_small))

In [ ]:
# Decode audio to waveform automatically
ds_audio_small = ds_audio_small.cast_column("audio", Audio(sampling_rate=44100))

# Check first sample
example = ds_audio_small[0]
waveform = example["audio"]["array"]
sr = example["audio"]["sampling_rate"]

print("Waveform shape:", waveform.shape, "Sampling rate:", sr)

In [ ]:
bundle = torchaudio.pipelines.SQUIM_OBJECTIVE

device = "cuda" if torch.cuda.is_available() else "cpu"

model = bundle.get_model().to(device)
model.eval()

print("SQUIM model loaded successfully on", device)

In [ ]:
def check_cough_quality_hf(waveform, sr, model, device="cuda"):
    """
    waveform: numpy array from Hugging Face 'audio' column
    sr: sample rate
    model: SQUIM model
    device: "cuda" or "cpu"
    """

    # Convert to tensor
    waveform_tensor = torch.tensor(waveform).float()

    # Convert to mono if needed
    if waveform_tensor.ndim > 1 and waveform_tensor.shape[0] > 1:
        waveform_tensor = waveform_tensor.mean(dim=0, keepdim=True)
    elif waveform_tensor.ndim == 1:
        waveform_tensor = waveform_tensor.unsqueeze(0)

    # Resample to 16kHz
    waveform_tensor = torchaudio.functional.resample(waveform_tensor, sr, 16000)

    # Move to device
    waveform_tensor = waveform_tensor.to(device)

    # Run SQUIM
    with torch.no_grad():
        stoi, pesq, si_sdr = model(waveform_tensor)

    return stoi.item(), pesq.item(), si_sdr.item()

In [ ]:
THRESHOLD_STOI = 0.8
THRESHOLD_PESQ = 2.0
THRESHOLD_SI_SDR = 10.0

In [ ]:
good_files = []
bad_files = []

for i, example in enumerate(ds_audio_small):

    audio_data = example["audio"]

    if audio_data is None or audio_data["array"] is None:
        print(f"Skipping sample {i}: audio is None")
        continue

    waveform = audio_data["array"]
    sr = audio_data["sampling_rate"]

    # Convert to tensor
    waveform_tensor = torch.tensor(waveform).float()

    if waveform_tensor.ndim == 1:
        waveform_tensor = waveform_tensor.unsqueeze(0)

    # Resample
    waveform_tensor = torchaudio.functional.resample(waveform_tensor, sr, 16000)

    waveform_tensor = waveform_tensor.to(device)

    # Run SQUIM
    with torch.no_grad():
        stoi, pesq, si_sdr = model(waveform_tensor)

    # Apply thresholds
    if pesq >= THRESHOLD_PESQ and stoi >= THRESHOLD_STOI and si_sdr >= THRESHOLD_SI_SDR:
        good_files.append(example)
    else:
        bad_files.append(example)

print("✅ Good files:", len(good_files))
print("❌ Bad files:", len(bad_files))

In [ ]:
os.makedirs("good_coughs", exist_ok=True)

for i, example in enumerate(good_files):

    waveform = example["audio"]["array"]
    sr = example["audio"]["sampling_rate"]

    sf.write(f"good_coughs/cough_{i}.wav", waveform, sr)

print("Saved all good coughs to folder 'good_coughs'")